In [1]:
import json
import numpy as np
import pandas as pd
import torch
#import geopandas as gpd

In [2]:
import sys

sys.path.append('../src')
from data_loader import load_demographics


/home/fpnerini/miniconda3/envs/pyg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
business_df = pd.read_csv(f'../datasets/yelp2019_business.csv')

business_df.loc[:, 'postal_code'] = business_df['postal_code'].fillna(0)
business_df['postal_code'] = business_df['postal_code'].astype(str)
business_df.loc[:, 'block_group_id'] = business_df['block_group_id'].fillna(0)
business_df['block_group_id'] = business_df['block_group_id'].astype(int).astype(str)

block_groups = business_df['block_group_id'].unique()


# converting full names into abbreviations
states_names_for_superres ={
    'Missouri': 'MO',
    'Pennsylvania': 'PA',
    'Tennessee': 'TN',
    'Florida': 'FL',
    'Indiana': 'IN',
    }


full_bg_df = load_demographics(states_names_in_dataset=states_names_for_superres, block_groups=block_groups, force_reload=True)

geoId/12 16697
geoId/18 6075
geoId/29 6165
geoId/42 11531
geoId/47 5377


In [25]:
variables = ['Median_HomeValue', 'Median_Income', 'Median_Age']

for var in variables:
    zip_bg_df = full_bg_df.groupby('zip')[[var, var+'_zip']].mean()
    # correlation between the two columns
    corr = zip_bg_df[var].corr(zip_bg_df[var+'_zip'])
    print(f'Correlation between {var} at BG and ZIP level: {corr:.4f}')
    # Mean percentage error between the two columns
    mape = np.mean(np.abs(zip_bg_df[var] - zip_bg_df[var+'_zip']) / zip_bg_df[var+'_zip']) * 100
    print(f'Mean Absolute Percentage Error between {var} at BG and ZIP level: {mape:.4f}\n')

Correlation between Median_HomeValue at BG and ZIP level: 0.9261
Mean Absolute Percentage Error between Median_HomeValue at BG and ZIP level: 13.6497

Correlation between Median_Income at BG and ZIP level: 0.8675
Mean Absolute Percentage Error between Median_Income at BG and ZIP level: 14.3715

Correlation between Median_Age at BG and ZIP level: 0.8636
Mean Absolute Percentage Error between Median_Age at BG and ZIP level: 7.5904



In [27]:
# now perform a weighted mean according to the Population variable
variables = ['Median_HomeValue', 'Median_Income', 'Median_Age']

for var in variables:
    curr_df = full_bg_df.copy()
    curr_df[var+'_weighted'] = curr_df[var] * curr_df['Population']
    curr_df[var+'_weighted'] = curr_df[var+'_weighted']

    zip_bg_df = curr_df.groupby('zip')[[var+'_weighted', 'Population']].sum()
    zip_bg_df[var] = zip_bg_df[var+'_weighted'] / zip_bg_df['Population']
    
    zip_bg_df[var+'_zip'] = curr_df.groupby('zip')[var+'_zip'].mean()
    # correlation between the two columns
    corr = zip_bg_df[var].corr(zip_bg_df[var+'_zip'])

    print(f'Weighted Correlation between {var} at BG and ZIP level: {corr:.4f}')
    # Mean percentage error between the two columns
    mape = np.mean(np.abs(zip_bg_df[var] - zip_bg_df[var+'_zip']) / zip_bg_df[var+'_zip']) * 100
    print(f'Weighted Mean Absolute Percentage Error between {var} at BG and ZIP level: {mape:.4f}\n')

Weighted Correlation between Median_HomeValue at BG and ZIP level: 0.9131
Weighted Mean Absolute Percentage Error between Median_HomeValue at BG and ZIP level: 15.0841

Weighted Correlation between Median_Income at BG and ZIP level: 0.8723
Weighted Mean Absolute Percentage Error between Median_Income at BG and ZIP level: 14.0419

Weighted Correlation between Median_Age at BG and ZIP level: 0.8701
Weighted Mean Absolute Percentage Error between Median_Age at BG and ZIP level: 6.7316



In [14]:

business_df = pd.read_csv(f'../datasets/yelp2019_business.csv')

business_df.loc[:, 'postal_code'] = business_df['postal_code'].fillna(0)
business_df['postal_code'] = business_df['postal_code'].astype(str)
business_df.loc[:, 'block_group_id'] = business_df['block_group_id'].fillna(0)
business_df['block_group_id'] = business_df['block_group_id'].astype(int).astype(str)

block_groups = business_df['block_group_id'].unique()

# converting full names into abbreviations
states_names_for_superres ={
    'Illinois': 'IL',
    'New Jersey': 'NJ',
    'Delaware': 'DE',
    'Missouri': 'MO',
    'Pennsylvania': 'PA',
    'Tennessee': 'TN',
    'Florida': 'FL',
    'Indiana': 'IN',
    }

full_bg_df = load_demographics(states_names_for_superres, block_groups)


dem_var_list = ['Median_Income', 'Median_HomeValue', 'Median_Age', ] #
dem_var_names = {'Median_HomeValue': 'Median Home Value', 'Median_Income': 'Median Income', 'Median_Age': 'Median Age'}

urban_areas = ['Philadelphia', 'St. Louis', 'Indianapolis', 'Nashville', 'Tampa']


for urbe in urban_areas:
    for dem_var_name in dem_var_list:

        curr_buss_df = business_df.loc[business_df.urban_area == urbe].copy()
        curr_bg = curr_buss_df.block_group_id.unique()
        
        # check that all the curr_bg are in full_bg_df
        curr_bg = [bg for bg in curr_bg if bg in full_bg_df.index]
        curr_buss_df = business_df.loc[business_df.block_group_id.isin(curr_bg)].copy()

        for target_entity in ['bg',  'tract']:
            if target_entity == 'bg':
                # drop any business whose block group target variable is null
                curr_buss_df = curr_buss_df[~curr_buss_df.block_group_id.isin(full_bg_df[full_bg_df[dem_var_name].isnull()].index)]
                curr_buss_bg = curr_buss_df.block_group_id.values

            else:
                # drop any business whose macroentity target variable is null
                curr_buss_df = curr_buss_df[~curr_buss_df.block_group_id.isin(full_bg_df[full_bg_df[f'{dem_var_name}_{target_entity}'].isnull()].index)]
                curr_buss_bg = curr_buss_df.block_group_id.values


        #curr_tracts_values = full_bg_df.loc[curr_bg, f'{dem_var_name}_tract'].values

        curr_buss_df = curr_buss_df[['longitude', 'latitude', ]].copy()

        #curr_buss_df.loc[:, 'target'] = full_bg_df.loc[curr_buss_bg, f'{dem_var_name}'].values


        curr_buss_df = curr_buss_df.dropna().copy()
        if len(curr_buss_df) == 0:
            print(f'No businesses in urban area {urbe} with non-null target variable {dem_var_name}, skipping')
            break

        num_datapoints = len(curr_buss_df)
        print(f'Urban Area: {urbe}, Target Variable: {dem_var_names[dem_var_name]}, Target Entity: {target_entity}, Num Data Points: {num_datapoints}')

Block groups demographics already downloaded
Urban Area: Philadelphia, Target Variable: Median Income, Target Entity: tract, Num Data Points: 29555
Urban Area: Philadelphia, Target Variable: Median Home Value, Target Entity: tract, Num Data Points: 29566
Urban Area: Philadelphia, Target Variable: Median Age, Target Entity: tract, Num Data Points: 30573
Urban Area: St. Louis, Target Variable: Median Income, Target Entity: tract, Num Data Points: 9174
Urban Area: St. Louis, Target Variable: Median Home Value, Target Entity: tract, Num Data Points: 8938
Urban Area: St. Louis, Target Variable: Median Age, Target Entity: tract, Num Data Points: 9395
Urban Area: Indianapolis, Target Variable: Median Income, Target Entity: tract, Num Data Points: 6145
Urban Area: Indianapolis, Target Variable: Median Home Value, Target Entity: tract, Num Data Points: 5731
Urban Area: Indianapolis, Target Variable: Median Age, Target Entity: tract, Num Data Points: 6198
Urban Area: Nashville, Target Variable: 